# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [2]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

You are already connected to a glueetl session af717e1e-1e58-452c-85e5-c12c98edb055.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.


You are already connected to a glueetl session af717e1e-1e58-452c-85e5-c12c98edb055.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 5.1


You are already connected to a glueetl session af717e1e-1e58-452c-85e5-c12c98edb055.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous worker type: None
Setting new worker type to: G.1X


You are already connected to a glueetl session af717e1e-1e58-452c-85e5-c12c98edb055.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: None
Setting new number of workers to: 5



In [2]:
s3_raw_path = "s3://transaction.dataset/raw/"

df_transactions = spark.read.parquet(s3_raw_path + "fTransactions/")
df_country = spark.read.parquet(s3_raw_path + "dCountry/")
df_product = spark.read.parquet(s3_raw_path + "dProduct/")

In [3]:
df_transactions.show(5)

+----------+--------------------+-------+--------+---------------+---------------+-----------+
|      Date|             Website|Product|Quantity|RevenueDiscount|NetStandardCost|CountryCode|
+----------+--------------------+-------+--------+---------------+---------------+-----------+
|2018-05-09|coloradoboomerang...|GelFast|      30|           0.15|           0.99|        NOR|
|2018-05-09|coloradoboomerang...|Fun Fly|      34|           0.15|           0.99|        KHM|
|2018-05-09|          amazon.com|Fun Fly|       2|          0.019|           0.99|        IRL|
|2018-05-09|coloradoboomerang...|Fun Fly|      59|          0.375|           0.99|        THA|
|2018-05-09|   gel-boomerang.com|   Quad|       2|              0|           0.99|        IND|
+----------+--------------------+-------+--------+---------------+---------------+-----------+
only showing top 5 rows


In [4]:
# Check schemas
print("Transactions schema:")
df_transactions.printSchema()

print("Country schema:")
df_country.printSchema()

print("Product schema:")
df_product.printSchema()

Transactions schema:
root
 |-- Date: date (nullable = true)
 |-- Website: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- RevenueDiscount: string (nullable = true)
 |-- NetStandardCost: decimal(10,2) (nullable = true)
 |-- CountryCode: string (nullable = true)

Country schema:
root
 |-- CountryCode: string (nullable = true)
 |-- Country: string (nullable = true)

Product schema:
root
 |-- Product: string (nullable = true)
 |-- Retail Price: decimal(12,2) (nullable = true)
 |-- Standard Cost: decimal(12,2) (nullable = true)
 |-- Category: string (nullable = true)


In [5]:
# Check row counts
print("Transactions rows:", df_transactions.count())
print("Country rows:", df_country.count())
print("Product rows:", df_product.count())

Transactions rows: 7742561
Country rows: 126
Product rows: 22


In [6]:
# Preview data
df_transactions.show(5)
df_country.show(5)
df_product.show(5)

+----------+--------------------+-------+--------+---------------+---------------+-----------+
|      Date|             Website|Product|Quantity|RevenueDiscount|NetStandardCost|CountryCode|
+----------+--------------------+-------+--------+---------------+---------------+-----------+
|2018-05-09|coloradoboomerang...|GelFast|      30|           0.15|           0.99|        NOR|
|2018-05-09|coloradoboomerang...|Fun Fly|      34|           0.15|           0.99|        KHM|
|2018-05-09|          amazon.com|Fun Fly|       2|          0.019|           0.99|        IRL|
|2018-05-09|coloradoboomerang...|Fun Fly|      59|          0.375|           0.99|        THA|
|2018-05-09|   gel-boomerang.com|   Quad|       2|              0|           0.99|        IND|
+----------+--------------------+-------+--------+---------------+---------------+-----------+
only showing top 5 rows

+-----------+--------------------+
|CountryCode|             Country|
+-----------+--------------------+
|        AFG|  

In [7]:
print("Transactions before:", df_transactions.count())
# Remove duplicate rows from transactions
df_transactions = df_transactions.dropDuplicates()

print("Transactions after:", df_transactions.count())

Transactions before: 7742561
Transactions after: 6331124


In [8]:
# Create Year and Month columns from the Date column

from pyspark.sql.functions import year, month, to_date, col

# Make sure Date is treated as a date type
df_transactions = df_transactions.withColumn(
    "Date",
    to_date(col("Date"))
)

# Add Year and Month columns
df_transactions = df_transactions \
    .withColumn("Year", year(col("Date"))) \
    .withColumn("Month", month(col("Date")))

# Preview result
df_transactions.show(10)

+----------+--------------------+--------------+--------+---------------+---------------+-----------+----+-----+
|      Date|             Website|       Product|Quantity|RevenueDiscount|NetStandardCost|CountryCode|Year|Month|
+----------+--------------------+--------------+--------+---------------+---------------+-----------+----+-----+
|2018-06-05|            ebay.com| Crested Beaut|       6|           0.05|           0.99|        KEN|2018|    6|
|2018-06-05|            ebay.com|       Fun Fly|       4|              0|           0.99|        RUS|2018|    6|
|2018-06-05|          amazon.com|       Carlota|       8|          0.071|           0.99|        NZL|2018|    6|
|2018-06-05|coloradoboomerang...|       Fun Fly|       6|           0.05|           0.99|        JPN|2018|    6|
|2018-06-05|          amazon.com|         Aspen|       2|          0.019|           0.99|        IND|2018|    6|
|2018-06-05|coloradoboomerang...|       Fun Fly|       2|              0|           0.99|       

In [9]:
from pyspark.sql.functions import col, regexp_replace

df_transactions = df_transactions.withColumn(
    "Website",
    regexp_replace(col("Website"), "\\..*", "")
)

In [10]:
print("Transactions", df_transactions.show())

+----------+------------------+----------------+--------+---------------+---------------+-----------+----+-----+
|      Date|           Website|         Product|Quantity|RevenueDiscount|NetStandardCost|CountryCode|Year|Month|
+----------+------------------+----------------+--------+---------------+---------------+-----------+----+-----+
|2018-06-05|              ebay|   Crested Beaut|       6|           0.05|           0.99|        KEN|2018|    6|
|2018-06-05|              ebay|         Fun Fly|       4|              0|           0.99|        RUS|2018|    6|
|2018-06-05|            amazon|         Carlota|       8|          0.071|           0.99|        NZL|2018|    6|
|2018-06-05|coloradoboomerangs|         Fun Fly|       6|           0.05|           0.99|        JPN|2018|    6|
|2018-06-05|            amazon|           Aspen|       2|          0.019|           0.99|        IND|2018|    6|
|2018-06-05|coloradoboomerangs|         Fun Fly|       2|              0|           0.99|       

In [11]:
# Join fact table with dimension tables
Sales_DataSet = df_transactions \
    .join(df_country, on="CountryCode", how="left") \
    .join(df_product, on="Product", how="left")

In [12]:
Sales_DataSet.show(10)

+--------------+-----------+----------+------------------+--------+---------------+---------------+----+-----+------------------+------------+-------------+------------+
|       Product|CountryCode|      Date|           Website|Quantity|RevenueDiscount|NetStandardCost|Year|Month|           Country|Retail Price|Standard Cost|    Category|
+--------------+-----------+----------+------------------+--------+---------------+---------------+----+-----+------------------+------------+-------------+------------+
| Crested Beaut|        KEN|2018-06-05|              ebay|       6|           0.05|           0.99|2018|    6|             Kenya|       19.95|         8.50|Intermediate|
|       Fun Fly|        RUS|2018-06-05|              ebay|       4|              0|           0.99|2018|    6|Russian Federation|        6.99|         3.25|    Beginner|
|       Carlota|        NZL|2018-06-05|            amazon|       8|          0.071|           0.99|2018|    6|       New Zealand|       21.95|        

In [13]:
# Save processed data to Amazon S3:
s3_base_path = "s3://transaction.dataset/processed/"

# 1) Save as Parquet
Sales_DataSet.write \
    .mode("overwrite") \
    .parquet(s3_base_path + "parquet/Sales_DataSet/")

print("Saved as Parquet")

# 2) Save as CSV (single file for Power BI)
Sales_DataSet.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(s3_base_path + "csv_single/Sales_DataSet/")

print("Saved as CSV (single file)")

Saved as Parquet
Saved as CSV (single file)
